# Database Table Views - Bid Documents

This notebook demonstrates the extraction quality of complex bid documents, showing three examples:
1. **Small table** - Award Letter with simple vendor information
2. **Medium table** - Bid Summary with nested bid structures
3. **Large table** - Bid Tabs with extensive item-by-item breakdowns

In [26]:
import psycopg2
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv()

# Connect to database
conn = psycopg2.connect(
    host=os.getenv('DB_HOST'),
    port=os.getenv('DB_PORT'),
    dbname=os.getenv('DB_NAME'),
    user=os.getenv('DB_USER'),
    password=os.getenv('DB_PASSWORD')
)

print("✓ Connected to RDS PostgreSQL database")

✓ Connected to RDS PostgreSQL database


## Example 1: Small Table - Award Letter (Simple Structure)
Award letters contain basic contract information with simple vendor data.

In [27]:
# Get all table names in the database
cursor = conn.cursor()
cursor.execute("""
    SELECT table_name 
    FROM information_schema.tables 
    WHERE table_schema = 'public' 
    ORDER BY table_name
""")

all_tables = [row[0] for row in cursor.fetchall()]
print(f"Total tables in database: {len(all_tables)}\n")
print("All tables:")
for table in all_tables:
    print(f"  - {table}")

Total tables in database: 24

All tables:
  - DA00539_Award_Letter.docx
  - DA00539_Bid_Tabs
  - DA00539_Bid_Tabs_bid_values
  - DA00539_Bid_Tabs_table
  - DA00539_Invitation_to_Bid
  - DA00540_Award_Letter
  - DA00540_Bid_Tabs
  - DA00540_Bid_Tabs_bid_values
  - DA00540_Bid_Tabs_table
  - DA00540_Invitation_to_Bid
  - DA00562_Award_Letter
  - DA00562_Bid_Tabs
  - DA00562_Bid_Tabs_bid_values
  - DA00562_Bid_Tabs_table
  - DA00562_Invitation_to_Bid
  - DA00564_Award_Letter
  - DA00564_Bid_Tabs
  - DA00564_Bid_Tabs_bid_values
  - DA00564_Bid_Tabs_table
  - DA00564_Invitation_to_Bid
  - L230201A_Item_C_Report
  - L230201A_Item_C_Report_vendor_names
  - L230215A_Item_C_Report
  - L230215A_Item_C_Report_vendor_names


## Example 2: Medium Table - Bid Summary (Nested Arrays)
Bid summary documents contain multiple bids with nested vendor names and bid amounts.

In [ ]:
# Query the first 10 records from DA00539_Bid_Tabs_bid_values
df_bid_values = pd.read_sql_query('SELECT * FROM "DA00562_Bid_Tabs_table" LIMIT 15', conn)
print(f"Table: DA00562_Bid_Tabs_table")
print(f"Showing first 10 records (Total columns: {len(df_bid_values.columns)})\n")
display(df_bid_values)

Table: DA00540_Bid_Tabs_table
Showing first 10 records (Total columns: 12)



C:\Users\anderson\AppData\Local\Temp\ipykernel_3704\2902879704.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_bid_values = pd.read_sql_query('SELECT * FROM "DA00540_Bid_Tabs_table" LIMIT 15', conn)


,id,da00540_bid_tabs_id,row_index,item_line_number,item_number,item_description,item_quantity,item_unit,vendor_1_unit_price,vendor_1_extended_price,vendor_2_unit_price,vendor_2_extended_price
0,1,1,0,1.0,0000100000-N 800,MOBILIZATION,1.0,Lump Sum,101000.00,101000.00,112000.00,112000.00
1,2,1,1,2.0,1803500000-E 660,"AST, DOUBLE SEAL",380471.0,SY,1.76,669628.96,1.71,650605.41
2,3,1,2,3.0,1838000000-E 660,EMULSION FOR AST,232657.0,GAL,2.50,581642.50,2.44,567683.08
3,4,1,3,4.0,4413000000-E SP,WORK ZONE ADV/GEN WARN SIGN,890.0,SF,7.00,6230.00,9.00,8010.00
4,5,1,4,5.0,4457000000-N SP,TEMP TRAFFIC CONTROL (SP),1.0,Lump Sum,28600.00,28600.00,76500.00,76500.00


## Example 3: Large Table - Bid Tabs (Complex Multi-Vendor Item Breakdown)
Bid tabs contain extensive item-by-item cost breakdowns with multiple vendors competing. These are the most complex documents with detailed line items, quantities, unit prices, and extended prices across multiple vendors.

In [29]:
# Find a Bid Tabs table (largest complexity)
cursor.execute("""
    SELECT table_name 
    FROM information_schema.tables 
    WHERE table_schema = 'public' 
    AND table_name LIKE '%Bid_Tabs%'
    ORDER BY table_name
    LIMIT 1
""")

bid_tabs_table = cursor.fetchone()[0]
print(f"Table: {bid_tabs_table}\n")

# Display main table
df = pd.read_sql_query(f'SELECT * FROM "{bid_tabs_table}"', conn)
print(f"Main table - Total records: {len(df)}")
print(f"Columns: {len(df.columns)}\n")
display(df)

# Check for related table (item breakdown)
related_table = f"{bid_tabs_table}_table"
cursor.execute(f"""
    SELECT EXISTS (
        SELECT FROM information_schema.tables 
        WHERE table_schema = 'public' 
        AND table_name = '{related_table}'
    )
""")

if cursor.fetchone()[0]:
    print(f"\nRelated table: {related_table}")
    df_items = pd.read_sql_query(f'SELECT * FROM "{related_table}"', conn)
    print(f"Total line items: {len(df_items)}")
    print(f"Columns per item: {len(df_items.columns)}\n")
    display(df_items)
else:
    print("\nNo related table found - data stored as JSONB in main table")

# Close connection
cursor.close()
conn.close()
print("\n✓ Connection closed")

Table: DA00539_Bid_Tabs

Main table - Total records: 1
Columns: 19

Main table - Total records: 1
Columns: 19



C:\Users\anderson\AppData\Local\Temp\ipykernel_3704\4020196037.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(f'SELECT * FROM "{bid_tabs_table}"', conn)


,id,document_type,source_file,created_at,county,project_number,proposal_project_type,location,contract_id,tip_no_,fed_aid_no,letting_date,call_number,vendor_names,total_vendor_1,total_vendor_2,miles,funding,timestamp
0,1,NC D1 Bid Tabs (Standard),2023 nc d1/2023-02-15_nc_d1/DA00539_Bid Tabs.pdf,2025-11-25 16:54:51.180651,"BERTIE, HERTFORD, HYDE, NORTHAMPTON, TYRRELL","[2026CPT.01.02.20082, 2026CPT.01.02.20462, 202...",ASPHALT SURFACE TREATMENT (DOUBLE SEAL),11 SECTION OF SECONDARY ROADS,DA00653,None,STATE FUNDED,2025-09-17,002,"[RILEY PAVING INC, WHITEHURST PAVING CO INC]",1387101.46,1414798.49,29.996,STATE FUNDED,2025-09-22T09:39:00



Related table: DA00539_Bid_Tabs_table
Total line items: 5
Columns per item: 12

Total line items: 5
Columns per item: 12



C:\Users\anderson\AppData\Local\Temp\ipykernel_3704\4020196037.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_items = pd.read_sql_query(f'SELECT * FROM "{related_table}"', conn)


,id,da00539_bid_tabs_id,row_index,item_line_number,item_number,item_description,item_quantity,item_unit,vendor_1_unit_price,vendor_1_extended_price,vendor_2_unit_price,vendor_2_extended_price
0,1,1,0,0001,0000100000-N 800,MOBILIZATION,Lump Sum,Lump Sum,101000.00,101000.00,112000.00,112000.00
1,2,1,1,0002,1803500000-E 660,"AST, DOUBLE SEAL",380471,SY,1.76,669628.96,1.71,650605.41
2,3,1,2,0003,1838000000-E 660,EMULSION FOR AST,232657,GAL,2.50,581642.50,2.44,567683.08
3,4,1,3,0004,4413000000-E SP,WORK ZONE ADV/GEN WARN SIGN,890,SF,7.00,6230.00,9.00,8010.00
4,5,1,4,0005,4457000000-N SP,TEMP TRAFFIC CONTROL (SP),Lump Sum,Lump Sum,28600.00,28600.00,76500.00,76500.00



✓ Connection closed
